# Task 2: Credit Card Fraud Detection
**CodSoft Machine Learning Internship**

**Goal:** Build a model that classifies credit card transactions as *fraudulent* or *legitimate*.

**Dataset:** [Credit Card Fraud Detection (Kaggle, ULB)](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud). Place `creditcard.csv` in the same folder as this notebook.

**Approach**
1. Explore the data and confirm how imbalanced it is
2. Preprocess (scale features, stratified train/test split)
3. Train Logistic Regression, Decision Tree and Random Forest
4. Compare ways of handling imbalance: nothing, class weights, SMOTE
5. Evaluate with precision, recall, F1 and PR-AUC (not accuracy)

## 1. Setup

In [ ]:
# Run once if needed:
# !pip install pandas numpy scikit-learn matplotlib seaborn imbalanced-learn

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                             precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score,
                             precision_recall_curve, ConfusionMatrixDisplay)
from imblearn.over_sampling import SMOTE

RANDOM_STATE = 42
sns.set_style("whitegrid")

## 2. Load the data

In [ ]:
df = pd.read_csv("creditcard.csv")
print("Shape:", df.shape)
df.head()

## 3. Exploratory Data Analysis

In [ ]:
print("Missing values:", df.isnull().sum().sum())
print("Duplicate rows:", df.duplicated().sum())
df.info()

In [ ]:
counts = df["Class"].value_counts()
print(counts)
print(f"\nFraud share: {counts[1] / len(df) * 100:.3f}%")

fig, ax = plt.subplots(figsize=(5, 4))
sns.countplot(x="Class", data=df, ax=ax)
ax.set_yscale("log")
ax.set_xticklabels(["Legit (0)", "Fraud (1)"])
ax.set_title("Class distribution (log scale)")
plt.show()

**Observation:** Fraud is a tiny fraction of all transactions. A model that always predicts "legit" would score about 99.8% accuracy while catching zero fraud, so **accuracy is a misleading metric here**. We will focus on precision, recall, F1 and PR-AUC for the fraud class.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df[df.Class == 0]["Amount"], bins=50, ax=axes[0], color="steelblue")
axes[0].set_title("Amount: legit"); axes[0].set_yscale("log")
sns.histplot(df[df.Class == 1]["Amount"], bins=50, ax=axes[1], color="crimson")
axes[1].set_title("Amount: fraud"); axes[1].set_yscale("log")
plt.show()

print(df.groupby("Class")["Amount"].describe())

In [ ]:
# Which features correlate most with fraud?
corr = df.corr()["Class"].drop("Class").sort_values()
fig, ax = plt.subplots(figsize=(8, 7))
corr.plot(kind="barh", ax=ax)
ax.set_title("Correlation of each feature with Class")
plt.show()

## 4. Preprocessing

`V1`-`V28` are already PCA-transformed and scaled, so only `Time` and `Amount` need scaling. We split first and fit the scaler on the **training set only**, to avoid data leakage. The split is **stratified** so train and test keep the same fraud ratio.

In [ ]:
df = df.drop_duplicates().reset_index(drop=True)

X = df.drop("Class", axis=1)
y = df["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

scaler = StandardScaler()
cols = ["Time", "Amount"]
X_train[cols] = scaler.fit_transform(X_train[cols])
X_test[cols] = scaler.transform(X_test[cols])

print("Train:", X_train.shape, "| fraud:", y_train.sum())
print("Test: ", X_test.shape,  "| fraud:", y_test.sum())

## 5. Evaluation helper

In [ ]:
results = []        # collects metrics for the final comparison
fitted = {}         # stores fitted models and test probabilities

def evaluate(name, model, X_te=None, y_te=None):
    X_te = X_test if X_te is None else X_te
    y_te = y_test if y_te is None else y_te
    pred = model.predict(X_te)
    proba = model.predict_proba(X_te)[:, 1]

    row = {
        "Model": name,
        "Precision": precision_score(y_te, pred, zero_division=0),
        "Recall": recall_score(y_te, pred),
        "F1": f1_score(y_te, pred),
        "ROC-AUC": roc_auc_score(y_te, proba),
        "PR-AUC": average_precision_score(y_te, proba),
    }
    results.append(row)
    fitted[name] = (model, proba)

    print(f"===== {name} =====")
    print(classification_report(y_te, pred, target_names=["Legit", "Fraud"], digits=4))
    ConfusionMatrixDisplay(confusion_matrix(y_te, pred),
                           display_labels=["Legit", "Fraud"]).plot(cmap="Blues", values_format="d")
    plt.title(name)
    plt.show()

## 6. Models

### 6.1 Baseline: Logistic Regression with no imbalance handling

In [ ]:
lr_base = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
lr_base.fit(X_train, y_train)
evaluate("LogReg (baseline)", lr_base)

### 6.2 Logistic Regression with `class_weight='balanced'`

In [ ]:
lr_cw = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE)
lr_cw.fit(X_train, y_train)
evaluate("LogReg (class weight)", lr_cw)

### 6.3 Logistic Regression with SMOTE
SMOTE creates synthetic fraud examples. It is applied to the **training set only**; the test set stays untouched.

In [ ]:
smote = SMOTE(random_state=RANDOM_STATE)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)
print("After SMOTE:", np.bincount(y_train_sm))

lr_sm = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
lr_sm.fit(X_train_sm, y_train_sm)
evaluate("LogReg (SMOTE)", lr_sm)

### 6.4 Decision Tree (class weight)

In [ ]:
dt = DecisionTreeClassifier(max_depth=8, min_samples_leaf=5,
                            class_weight="balanced", random_state=RANDOM_STATE)
dt.fit(X_train, y_train)
evaluate("Decision Tree (class weight)", dt)

### 6.5 Random Forest (class weight)

In [ ]:
rf = RandomForestClassifier(n_estimators=100, min_samples_leaf=2,
                            class_weight="balanced_subsample",
                            n_jobs=-1, random_state=RANDOM_STATE)
rf.fit(X_train, y_train)
evaluate("Random Forest (class weight)", rf)

### 6.6 Random Forest with SMOTE
(This one takes a little longer because SMOTE doubles the training set.)

In [ ]:
rf_sm = RandomForestClassifier(n_estimators=100, min_samples_leaf=2,
                               n_jobs=-1, random_state=RANDOM_STATE)
rf_sm.fit(X_train_sm, y_train_sm)
evaluate("Random Forest (SMOTE)", rf_sm)

## 7. Model comparison

In [ ]:
res = (pd.DataFrame(results).set_index("Model")
         .sort_values("PR-AUC", ascending=False).round(4))
res

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for name, (_, proba) in fitted.items():
    p, r, _ = precision_recall_curve(y_test, proba)
    ax.plot(r, p, label=f"{name} (AP={average_precision_score(y_test, proba):.3f})")
ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
ax.set_title("Precision-Recall curves")
ax.legend(fontsize=8, loc="lower left")
plt.show()

## 8. Feature importance (Random Forest)

In [ ]:
imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values().tail(15)
fig, ax = plt.subplots(figsize=(7, 5))
imp.plot(kind="barh", ax=ax)
ax.set_title("Top 15 features (Random Forest)")
plt.show()

## 9. Threshold tuning (optional)
By default a model flags fraud when probability > 0.5. In fraud detection the right trade-off between catching fraud (recall) and false alarms (precision) is a business decision, so we can move the threshold.

In [ ]:
best_name = res.index[0]
_, best_proba = fitted[best_name]
print("Best model by PR-AUC:", best_name)

rows = []
for t in [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
    pred_t = (best_proba >= t).astype(int)
    rows.append({"Threshold": t,
                 "Precision": precision_score(y_test, pred_t, zero_division=0),
                 "Recall": recall_score(y_test, pred_t),
                 "F1": f1_score(y_test, pred_t)})
pd.DataFrame(rows).round(4)

## 10. Conclusion

- The data is **extremely imbalanced**, so accuracy is not a useful metric; I compared models using precision, recall, F1 and PR-AUC on the fraud class.
- The plain Logistic Regression baseline reaches high precision but misses more fraud. Class weights and SMOTE raise recall, at the cost of many more false alarms for Logistic Regression.
- **Tree ensembles (Random Forest)** usually give the best balance between precision and recall on this dataset. Check the comparison table above for the exact numbers from your run.
- **Possible improvements:** try Gradient Boosting / XGBoost, tune hyperparameters with cross-validation, and choose the decision threshold based on the cost of a missed fraud versus a false alarm.